In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p /content/local_data/train
!mkdir -p /content/local_data/val

In [ ]:
# Copy compressed archives from Drive to lightning fast Colab runtime storage
!cp /content/drive/MyDrive/Argoverse_Project/data/forecasting_train_v1.1.tar.gz /content/local_data/
!cp /content/drive/MyDrive/Argoverse_Project/data/forecasting_val_v1.1.tar.gz /content/local_data/

In [ ]:
# extracting files to colab runtime storage
!tar -xzf /content/local_data/forecasting_train_v1.1.tar.gz -C /content/local_data/train --strip-components=1
!tar -xzf /content/local_data/forecasting_val_v1.1.tar.gz -C /content/local_data/val --strip-components=1
print("✅ Extraction complete!")

✅ Extraction complete!


In [ ]:
os.chdir('/content/drive/MyDrive/Argoverse_Project/scripts')

In [ ]:
if not os.path.exists('../data'):
  !mkdir -p ../data
!ln -s /content/local_data/train/data ../data/train
!ln -s /content/local_data/val/data ../data/val


In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from data_loader import load_sequence  # Uses your exact custom load function

def pre_process_dataset(source_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(source_folder) if f.endswith('.csv')]

    print(f"Converting {len(files)} files from {source_folder} to binary tensors...")
    for f in tqdm(files):
        file_path = os.path.join(source_folder, f)
        data = load_sequence(file_path)
        if data is not None:
            past, future, origin, theta = data
            # Package them into a single dictionary tensor
            payload = {
                'past': torch.tensor(past, dtype=torch.float32),
                'future': torch.tensor(future, dtype=torch.float32),
                'origin': torch.tensor(origin, dtype=torch.float32),
                'theta': torch.tensor(theta, dtype=torch.float32)
            }
            # Save as binary file
            output_path = os.path.join(output_folder, f.replace('.csv', '.pt'))
            torch.save(payload, output_path)

# Convert both splits on Colab's fast scratch disk
pre_process_dataset("/content/local_data/train/data", "/content/tensor_data/train")
pre_process_dataset("/content/local_data/val/data", "/content/tensor_data/val")

Converting 205942 files from /content/local_data/train/data to binary tensors...


100%|██████████| 205942/205942 [21:14<00:00, 161.54it/s]


Converting 39472 files from /content/local_data/val/data to binary tensors...


100%|██████████| 39472/39472 [04:03<00:00, 161.86it/s]


In [ ]:
!python train.py

 Training data loaded in 0.51 [s]
0.5121369361877441
 Validation data loaded in 0.61 [s]
Device Name: Tesla T4
looping starts in 3.25 s
Epoch: 0 | Train Loss: 3.0993 | Val Loss: 1.7515 | ADE: 3.56m | FDE: 7.71m | Time: 212.39s
Epoch: 1 | Train Loss: 1.4415 | Val Loss: 1.3110 | ADE: 2.74m | FDE: 6.06m | Time: 154.40s
Epoch: 2 | Train Loss: 1.3323 | Val Loss: 1.1586 | ADE: 2.49m | FDE: 5.53m | Time: 153.83s
Epoch: 3 | Train Loss: 1.1619 | Val Loss: 0.9370 | ADE: 2.10m | FDE: 4.77m | Time: 156.46s
Epoch: 4 | Train Loss: 1.0184 | Val Loss: 0.8866 | ADE: 2.00m | FDE: 4.55m | Time: 153.74s
Epoch: 5 | Train Loss: 0.9728 | Val Loss: 0.8288 | ADE: 1.91m | FDE: 4.33m | Time: 153.63s
Epoch: 6 | Train Loss: 0.9494 | Val Loss: 0.8362 | ADE: 1.93m | FDE: 4.36m | Time: 152.38s
Epoch: 7 | Train Loss: 0.9350 | Val Loss: 0.8116 | ADE: 1.89m | FDE: 4.25m | Time: 153.30s
Epoch: 8 | Train Loss: 0.9261 | Val Loss: 0.7890 | ADE: 1.84m | FDE: 4.18m | Time: 154.35s
Epoch: 9 | Train Loss: 0.9134 | Val Loss: 0.7

In [ ]:
# Copy tensor data
# !mkdir -p /content/drive/MyDrive/Argoverse_Project/data/tensor_data
!tar -czf /content/tensor_data.tar.gz -C /content tensor_data
!cp /content/tensor_data.tar.gz /content/drive/MyDrive/Argoverse_Project/data/

In [ ]:
# testing
!mkdir -p /content/local_data/test
!cp /content/drive/MyDrive/Argoverse_Project/data/forecasting_test_v1.1.tar.gz /content/local_data/
!tar -xzf /content/local_data/forecasting_test_v1.1.tar.gz -C /content/local_data/test --strip-components=1